In [1]:
import time
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

df = sns.load_dataset('taxis').dropna()

cat_cols = ['color', 'pickup_borough', 'pickup_zone']
num_cols = ['passengers', 'distance', 'fare', 'tip', 'tolls']
target = (df['payment'] == 'credit card').astype(int)

for c in cat_cols:
    print(c, '->', df[c].nunique(), 'categories')

color -> 2 categories
pickup_borough -> 4 categories
pickup_zone -> 194 categories


In [2]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df[num_cols + cat_cols], target, test_size=0.2, random_state=42, stratify=target
)

def run_strategy(encoder, name):
    if encoder == 'label':
        X_train = X_train_raw.copy()
        X_test = X_test_raw.copy()
        for c in cat_cols:
            le = LabelEncoder()
            le.fit(pd.concat([X_train[c], X_test[c]]))
            X_train[c] = le.transform(X_train[c])
            X_test[c] = le.transform(X_test[c])
        feature_count = X_train.shape[1]
    else:
        preprocessor = ColumnTransformer([
            ('num', StandardScaler(), num_cols),
            ('cat', encoder, cat_cols)
        ])
        X_train = preprocessor.fit_transform(X_train_raw)
        X_test = preprocessor.transform(X_test_raw)
        feature_count = X_train.shape[1]

    model = LogisticRegression(max_iter=2000)
    start = time.perf_counter()
    model.fit(X_train, y_train)
    train_time = time.perf_counter() - start

    acc = accuracy_score(y_test, model.predict(X_test))
    return {'strategy': name, 'accuracy': acc, 'train_time_s': train_time, 'feature_count': feature_count}

In [3]:
results = []
results.append(run_strategy('label', 'Label Encoding'))
results.append(run_strategy(OneHotEncoder(handle_unknown='ignore'), 'One-Hot Encoding'))
results.append(run_strategy(OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), 'Ordinal Encoding'))

results_df = pd.DataFrame(results)
results_df

,strategy,accuracy,train_time_s,feature_count
0,Label Encoding,0.946414,0.058435,8
1,One-Hot Encoding,0.955083,0.039147,195
2,Ordinal Encoding,0.946414,0.051910,8
